# `parse_pdf` + `apply_changes` — pattern extract / modify / rebuild côté PDF

Module testé : [`src/docpipeline/parsing/pdf/parse_pdf.py`](../src/docpipeline/parsing/pdf/parse_pdf.py).

## Le pattern transversal du projet

Symétrique de `parse_word` côté PDF :

```
extract  : parse_pdf(pdf_in)              → line_df avec span_id stable + bbox + style
modify   : on construit {span_id: nouveau_texte}
rebuild  : apply_changes(pdf_in, span_changes, pdf_out)
           → reconstruit le PDF en redact + insert_textbox sur les bbox d'origine
```

Le `span_id` (format `p_<page>_<line>`) est la clé stable, cohérente avec le `w_<para>_<run>` côté Word.

## Limitations PDF (vs Word)

PyMuPDF ne permet de réinjecter du texte qu'avec les **Standard 14 fonts** (Helvetica, Times, Courier). Les polices custom du PDF d'origine ne peuvent pas être réembed-ées sans fournir le `.ttf`. Position / taille / couleur / gras / italique sont préservés, le rendu visuel des polices spécifiques peut différer.

## Démo : cas d'usage **traduction FR → EN**

On parse `tests/fixtures/notice_garanties.pdf` (notice d'assurance en français), on traduit chaque ligne via un mini-dictionnaire FR → EN (sans LLM pour la démo — en prod ce serait un appel LLM, autorisé par `CLAUDE.md` dans la couche translation), et on reconstruit le PDF.

Le PDF traduit est sauvegardé dans `notebooks/_outputs/pdf_translation/notice_garanties_EN_js.pdf` pour inspection visuelle.

In [1]:
# À exécuter une seule fois : installer le package en mode editable
import sys
!{sys.executable} -m pip install -q -e ..

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: C:\Users\DELL\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown
from docpipeline.parsing.pdf.parse_pdf import parse_pdf, apply_changes

PDF_FR = Path('../tests/fixtures/notice_garanties.pdf')
PDF_EN = Path('_outputs/pdf_translation/notice_garanties_EN_js.pdf')
PDF_EN.parent.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_colwidth', 120)

# 1) EXTRACT --------------------------------------------------------------
result = parse_pdf(PDF_FR)

display(Markdown('### 1. `doc_summary` — synthèse au niveau document'))
summary_df = pd.DataFrame([{'key': k, 'value': str(v)} for k, v in result['doc_summary'].items()])
display(summary_df)

display(Markdown(f"### 2. `line_df` — {len(result['line_df'])} lignes extraites avec leurs styles"))
display(result['line_df'][['span_id', 'page_num', 'text', 'font_name', 'font_size', 'bold', 'italic', 'color']])

Consider using the pymupdf_layout package for a greatly improved page layout analysis.


### 1. `doc_summary` — synthèse au niveau document

,key,value
0,pdf_hash,dd01e383e6e578d16ee67cb42824fada2c0b935fccaadfb0875b2a715938e32a
1,file_size_bytes,2572
2,n_pages,1
3,source_category,other
4,source_tool,Unknown
5,source_confidence,0.3
6,source_signals,['no_metadata']
7,creator_raw,
8,producer_raw,
9,pdf_version,1.7


### 2. `line_df` — 9 lignes extraites avec leurs styles

,span_id,page_num,text,font_name,font_size,bold,italic,color
0,p_0_0,0,Notice d'Information · Garanties,Helvetica,18.0,False,False,#000000
1,p_0_1,0,Ce document présente les garanties souscrites dans le cadre du contrat.,Helvetica,11.0,False,False,#000000
2,p_0_2,0,La garantie IA (Individual Accident) couvre tout sinistre corporel.,Helvetica,11.0,False,False,#000000
3,p_0_3,0,La garantie BI (Business Interruption) couvre les pertes d'exploitation.,Helvetica,11.0,False,False,#000000
4,p_0_4,0,Tableau des garanties :,Helvetica,13.0,False,False,#000000
5,p_0_5,0,Garantie Plafond Franchise,Helvetica,10.0,False,False,#000000
6,p_0_6,0,IA · Individual Accident 50 000 EUR 300 EUR,Helvetica,10.0,False,False,#000000
7,p_0_7,0,BI · Business Interruption 100 000 EUR 1 000 EUR,Helvetica,10.0,False,False,#000000
8,p_0_8,0,RC · Responsabilite Civile 200 000 EUR 0 EUR,Helvetica,10.0,False,False,#000000


In [3]:
# 2) MODIFY : traduction FR -> EN via dictionnaire metier (zero LLM pour la demo)
FR_TO_EN = {
    "Notice d'Information \u00b7 Garanties":
        "Information Notice \u00b7 Coverage",
    "Ce document pr\u00e9sente les garanties souscrites dans le cadre du contrat.":
        "This document presents the coverage subscribed under the contract.",
    "La garantie IA (Individual Accident) couvre tout sinistre corporel.":
        "The IA (Individual Accident) coverage covers all bodily injury claims.",
    "La garantie BI (Business Interruption) couvre les pertes d'exploitation.":
        "The BI (Business Interruption) coverage covers operating losses.",
    "Tableau des garanties :":
        "Coverage table:",
    "Garantie                       Plafond         Franchise":
        "Coverage                       Limit           Deductible",
    "IA \u00b7 Individual Accident       50 000 EUR      300 EUR":
        "IA \u00b7 Individual Accident       50,000 EUR      300 EUR",
    "BI \u00b7 Business Interruption     100 000 EUR     1 000 EUR":
        "BI \u00b7 Business Interruption     100,000 EUR    1,000 EUR",
    "RC \u00b7 Responsabilite Civile     200 000 EUR     0 EUR":
        "RC \u00b7 Civil Liability          200,000 EUR    0 EUR",
}

ld = result['line_df']
changes = {row['span_id']: FR_TO_EN[row['text']]
           for _, row in ld.iterrows()
           if row['text'] in FR_TO_EN}

display(Markdown(f"### 3. Traductions appliquées : {len(changes)} / {len(ld)} spans"))
lines_idx = ld.set_index('span_id')
translations_df = pd.DataFrame([
    {'span_id': sid, 'FR (original)': lines_idx.loc[sid, 'text'], 'EN (traduit)': tr}
    for sid, tr in changes.items()
])
display(translations_df)

# 3) REBUILD
res = apply_changes(PDF_FR, changes, PDF_EN)
display(Markdown(f"**Fichier traduit écrit** : `{PDF_EN}` ({PDF_EN.stat().st_size} bytes)"))
display(Markdown(f"- spans replaced : **{res['spans_replaced']}**"))
display(Markdown(f"- spans skipped : **{res['spans_skipped']}**"))
if res['warnings']:
    display(Markdown(f"- warnings : {res['warnings']}"))

### 3. Traductions appliquées : 9 / 9 spans

,span_id,FR (original),EN (traduit)
0,p_0_0,Notice d'Information · Garanties,Information Notice · Coverage
1,p_0_1,Ce document présente les garanties souscrites dans le cadre du contrat.,This document presents the coverage subscribed under the contract.
2,p_0_2,La garantie IA (Individual Accident) couvre tout sinistre corporel.,The IA (Individual Accident) coverage covers all bodily injury claims.
3,p_0_3,La garantie BI (Business Interruption) couvre les pertes d'exploitation.,The BI (Business Interruption) coverage covers operating losses.
4,p_0_4,Tableau des garanties :,Coverage table:
5,p_0_5,Garantie Plafond Franchise,Coverage Limit Deductible
6,p_0_6,IA · Individual Accident 50 000 EUR 300 EUR,"IA · Individual Accident 50,000 EUR 300 EUR"
7,p_0_7,BI · Business Interruption 100 000 EUR 1 000 EUR,"BI · Business Interruption 100,000 EUR 1,000 EUR"
8,p_0_8,RC · Responsabilite Civile 200 000 EUR 0 EUR,"RC · Civil Liability 200,000 EUR 0 EUR"


**Fichier traduit écrit** : `_outputs\pdf_translation\notice_garanties_EN_js.pdf` (21691 bytes)

- spans replaced : **9**

- spans skipped : **0**

In [4]:
# 4) VERIF : on re-parse le PDF traduit et on compare
result_en = parse_pdf(PDF_EN)
before = result['line_df']
after  = result_en['line_df']

display(Markdown('### 4. Comparaison FR / EN — toutes les lignes côte à côte'))
n = min(len(before), len(after))
compare_df = pd.DataFrame({
    'span_id':       before['span_id'].head(n).values,
    'FR (avant)':    before['text'].head(n).values,
    'EN (après)':    after['text'].head(n).values,
})
display(compare_df)

display(Markdown('### 5. Bbox et font_size préservés ?'))
checks = []
for col in ['bbox', 'font_size']:
    same = sum(1 for i in range(n) if str(before.iloc[i][col]) == str(after.iloc[i][col]))
    checks.append({'attribute': col, 'matches': f'{same} / {n}'})
display(pd.DataFrame(checks))

display(Markdown(f"\n>>> Ouvre `{PDF_EN}` dans Adobe Reader / Sumatra pour valider visuellement la traduction."))

### 4. Comparaison FR / EN — toutes les lignes côte à côte

,span_id,FR (avant),EN (après)
0,p_0_0,Notice d'Information · Garanties,Information Notice · Coverage
1,p_0_1,Ce document présente les garanties souscrites dans le cadre du contrat.,This document presents the coverage subscribed under the contract.
2,p_0_2,La garantie IA (Individual Accident) couvre tout sinistre corporel.,The IA (Individual Accident) coverage covers all bodily injury claims.
3,p_0_3,La garantie BI (Business Interruption) couvre les pertes d'exploitation.,The BI (Business Interruption) coverage covers operating losses.
4,p_0_4,Tableau des garanties :,Coverage table:
5,p_0_5,Garantie Plafond Franchise,Coverage Limit Deductible
6,p_0_6,IA · Individual Accident 50 000 EUR 300 EUR,"IA · Individual Accident 50,000 EUR 300 EUR"
7,p_0_7,BI · Business Interruption 100 000 EUR 1 000 EUR,"BI · Business Interruption 100,000 EUR 1,000 EUR"
8,p_0_8,RC · Responsabilite Civile 200 000 EUR 0 EUR,"RC · Civil Liability 200,000 EUR 0 EUR"


### 5. Bbox et font_size préservés ?

,attribute,matches
0,bbox,0 / 9
1,font_size,0 / 9



>>> Ouvre `_outputs\pdf_translation\notice_garanties_EN_js.pdf` dans Adobe Reader / Sumatra pour valider visuellement la traduction.